# 예제 01. 학과 LLM 서버에 처음 접속하기 · Example 01 — Connecting to the Department LLM Server

전산실험 2 · AI Agent 만들기 · **1주차** · 예상 소요 30분
Computational Experiments 2 · Building AI Agents · **Week 1** · about 30 minutes

## 목표 · Objectives
- 학과 서버에 어떤 모델이 올라가 있는지 확인한다
- 첫 응답을 받아낸다
- `messages` 의 세 가지 역할을 이해한다
- 모델이 **이전 대화를 기억하지 못한다**는 사실을 직접 확인한다
- 응답에 토큰이 몇 개 쓰였는지 확인한다

> **EN**
> - Find out which models the department server is running
> - Get your first response back
> - Understand the three roles in `messages`
> - See for yourself that **the model does not remember the previous turn**
> - Check how many tokens a response consumed

> 셀을 클릭하고 **Shift + Enter** 로 실행합니다.
> **EN** — Click a cell and press **Shift + Enter** to run it.

---
## 0. 준비 — API 키를 안전하게 넣기 · Setup — Storing your API key safely

수업시간에 공개된 API 키를 **Colab Secrets** 에 저장합니다.

1. 왼쪽 세로 메뉴에서 **열쇠 아이콘(🔑)** 을 클릭
2. **새 secret 추가** — 두 개를 만듭니다
   - `KMU_BASE_URL` = 수업시간에 공개하는 **서버 주소**
   - `KMU_API_KEY` = 수업시간에 공개하는 **API 키**
3. 각각 이 노트북의 **액세스 토글을 켜기** ← 이걸 안 켜면 못 읽습니다

> 서버 주소도 키와 똑같이 관리합니다. 이 저장소는 공개되어 있으므로
> 주소를 코드에 적으면 외부에서 학과 서버를 두드릴 수 있게 됩니다.

키를 코드 셀에 직접 적으면 안 되는 이유는 2주차에 자세히 다룹니다.

> **EN** — Store the API key announced in class in **Colab Secrets**.
> 1. Click the **key icon (🔑)** in the left sidebar
> 2. **Add a new secret** — two of them:
>    - `KMU_BASE_URL` = the **server URL** given in class
>    - `KMU_API_KEY` = the **API key** given in class
> 3. Turn on **notebook access** for each one ← without this they cannot be read
>
> Treat the server URL exactly like the key. This repository is public, so
> writing the URL in code lets anyone outside reach the department server.
> Week 2 covers in detail why keys must never be typed into a code cell.

In [ ]:
# Colab에 openai 패키지 설치 (이미 있으면 그냥 넘어갑니다)
# EN — Install the openai package in Colab (skipped if already present)
!pip install -q openai
print("설치 완료")

In [ ]:
# 서버 주소와 API 키 가져오기
#   1순위: Colab Secrets (왼쪽 열쇠 아이콘)
#   2순위: 환경변수
#   3순위: 직접 입력
#
# 주의: 아래에 주소나 키를 직접 타이핑하지 마세요.
#       이 저장소는 공개되어 있어, 적는 순간 전 세계에 노출됩니다.
#
# EN — Fetch the server URL and the API key, in order of preference:
#        1st Colab Secrets (key icon on the left), 2nd environment variable,
#        3rd type it in.
#      WARNING: never type the URL or the key below. This repository is public,
#      so the moment you write it there, the whole world has it.

import os

def get_secret(name, secret=True):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            print(f"{name}: Colab Secrets 에서 가져왔습니다.")
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        print(f"{name}: 환경변수에서 가져왔습니다.")
        return os.environ[name].strip()

    print(f"[{name}] 를 Secrets 에서 찾지 못했습니다. 직접 입력해 주세요.")
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    # 주소를 통째로(.../chat/completions) 저장했어도 알아서 정리합니다
    # EN — trims the path even if you saved the full .../chat/completions URL
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


BASE_URL = get_base_url()
API_KEY  = get_secret("KMU_API_KEY")
MODEL    = "gemma4"

print("주소 길이:", len(BASE_URL), "글자  |  키 길이:", len(API_KEY), "글자")
print("(값 자체는 출력하지 않습니다)")

---
## 1. 서버 진단 — 질문하기 전에 먼저 확인한다 · Diagnosing the server before you ask anything

바로 질문을 던지지 않고, **가장 단순한 요청**부터 보냅니다.

`GET /v1/models` 는 "이 서버에 어떤 모델이 있나요?" 라고 묻는 요청입니다.
응답이 오면 주소와 키가 모두 정상이라는 뜻입니다.

처음 쓰는 API에서 오류가 났을 때, 주소가 틀린 건지 키가 틀린 건지 모델 이름이
틀린 건지 구분하지 못하면 시간을 크게 낭비합니다.
**되는 것부터 확인하고 하나씩 올라가는 것**이 요령입니다.

> **EN** — Do not fire off a question yet; send **the simplest possible request**
> first. `GET /v1/models` asks "which models does this server have?", and a
> response means both the URL and the key are fine.
>
> When an unfamiliar API errors out, being unable to tell a wrong URL from a
> wrong key from a wrong model name costs a great deal of time. The trick is to
> **confirm what works and climb up one step at a time.**

In [ ]:
import requests

# BASE_URL, API_KEY, MODEL 은 위 셀에서 Secrets 로부터 읽었습니다.
# EN — BASE_URL, API_KEY and MODEL were read from Secrets in the cell above.
response = requests.get(
    f"{BASE_URL}/models",
    headers={"Authorization": f"Bearer {API_KEY}"},
    timeout=15,
)

print("HTTP 상태코드:", response.status_code)
print(response.text[:500])

In [ ]:
# 응답을 보기 좋게 정리 / Tidy up the response for reading
if response.status_code == 200:
    models = [m["id"] for m in response.json()["data"]]
    print(f"사용 가능한 모델 {len(models)}개")
    for name in models:
        mark = "   <-- 우리가 쓸 모델" if name == MODEL else ""
        print("  -", name, mark)

    if MODEL not in models:
        print()
        print("[주의] MODEL 값이 목록에 없습니다.")
        print("       위 목록에서 정확한 이름을 골라 MODEL 을 고쳐 주세요.")
else:
    print("실패했습니다. 아래 셀의 안내를 읽어 보세요.")

### 연결이 안 될 때 · When the connection fails

| 증상 | 원인 | 해볼 것 |
|---|---|---|
| 연결 자체가 안 됨 | 주소·포트·프로토콜 | 아래 셀로 후보를 자동 탐색 |
| `401` / `403` | 키 문제 | Secrets 이름이 `KMU_API_KEY` 인지, 액세스 토글이 켜졌는지 |
| `404` | 경로 문제 | `KMU_BASE_URL` 끝이 `/v1` 인지 확인 |

**EN**

| Symptom | Cause | What to try |
|---|---|---|
| No connection at all | URL, port or protocol | Sweep the candidates in the cell below |
| `401` / `403` | Key problem | Check the secret is named `KMU_API_KEY` and its access toggle is on |
| `404` | Path problem | Check that `KMU_BASE_URL` ends in `/v1` |

아래 셀에서 주소 후보를 한꺼번에 시험해 볼 수 있습니다.

> **EN** — The cell below tries all the candidate URLs at once.

In [ ]:
# 주소 후보 자동 탐색 (연결이 안 될 때만 실행하세요)
# Secrets 에 넣은 주소를 조금씩 바꿔 가며 시험합니다.
# EN — Sweep candidate URLs; run this ONLY when the connection fails. It varies
#      the URL you put in Secrets a little at a time and tries each one.
from urllib.parse import urlparse

host = urlparse(BASE_URL).hostname or ""
candidates = [BASE_URL]
for scheme in ("https", "http"):
    for port in ("", ":11434", ":8000"):
        candidates.append(f"{scheme}://{host}{port}/v1")

for url in dict.fromkeys(candidates):            # 중복 제거, 순서 유지 / dedupe, keep order
    표시 = url.replace(host, host[:4] + "***")    # 전체 주소를 화면에 남기지 않는다 / never show the full URL
    try:
        r = requests.get(f"{url}/models",
                         headers={"Authorization": f"Bearer {API_KEY}"},
                         timeout=5)
        print(f"{r.status_code}  {표시}")
    except Exception as exc:
        print(f"실패  {표시}  ({type(exc).__name__})")

---
## 2. 첫 호출 · Your first call

이제 실제로 질문을 던집니다.

서버가 **OpenAI 호환** 규격이므로, OpenAI에서 만든 파이썬 라이브러리를
그대로 쓸 수 있습니다. 바꾸는 것은 `base_url` 하나뿐입니다.

> **EN** — Now we actually ask a question. The server speaks the
> **OpenAI-compatible** protocol, so OpenAI's own Python library works as is —
> the only thing you change is `base_url`.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=120)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "표준편차를 한 문장으로 설명해 주세요."}
    ],
)

print(response.choices[0].message.content)

### 방금 무슨 일이 일어났나 · What just happened

```
우리     →  {"model": ..., "messages": [...]}   →  서버
서버     →  {"choices": [{"message": {...}}], "usage": {...}}  →  우리
```

`response.choices[0].message.content` 라는 긴 경로를 매번 쓰기는 번거롭습니다.
그래서 함수로 감싸 둡니다. **다음 주에 이 함수들을 모아 파일 하나로 만들 것입니다.**

> **EN** — We send a model name and a message list; the server sends back
> `choices` and `usage`. Writing out the long path
> `response.choices[0].message.content` every time is tedious, so we wrap it in
> a function. **Next week we collect these functions into a single file.**

In [ ]:
def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL, messages=messages, **kw
    )
    return response.choices[0].message.content.strip()


print(ask("t검정을 한 문장으로 설명해 주세요."))

---
## 3. `messages` 의 세 가지 역할 · The three roles in `messages`

| 역할 | 뜻 | 언제 쓰나 |
|---|---|---|
| `system` | 모델의 성격·규칙 | 대화 전체에 적용할 지시 |
| `user` | 사용자의 말 | 질문, 요청 |
| `assistant` | 모델이 **앞서 한 말** | 대화를 이어 붙일 때 |

**EN**

| Role | Meaning | When to use it |
|---|---|---|
| `system` | The model's persona and rules | Instructions for the whole conversation |
| `user` | What the user said | Questions, requests |
| `assistant` | What the model **said earlier** | Stitching a conversation together |

`system` 을 바꾸면 같은 질문에도 답이 달라집니다.

> **EN** — Change `system` and the same question gets a different answer.

In [ ]:
질문 = "p값이 0.03이면 어떻게 해석하나요?"

for 역할설명 in [
    "너는 통계학과 대학원 조교다. 전문 용어를 그대로 쓰고 두 문장으로 답한다.",
    "너는 통계를 처음 배우는 학부생에게 설명하는 사람이다. 비유를 써서 쉽게 답한다.",
]:
    print("=" * 60)
    print("[system]", 역할설명)
    print("-" * 60)
    print(ask(질문, system=역할설명))
    print()

---
## 4. 모델은 이전 대화를 기억하지 못한다 · The model does not remember the previous turn

이것이 오늘 가장 중요한 내용입니다.

서버는 **우리가 보낸 메시지 목록만** 보고 답합니다.
대화가 이어지는 것처럼 보이는 이유는, 우리가 지난 대화를 매번 다시 보내기 때문입니다.

직접 확인해 봅시다.

> **EN** — This is the most important point today. The server answers from
> **the message list we send, and nothing else**. A conversation appears to
> continue only because we resend the past turns every time. Let's verify it.

In [ ]:
# (1) 이전 대화를 보내지 않으면 -- 모델은 아무것도 모릅니다
# EN — (1) Send no history, and the model knows nothing
답1 = ask("제 이름은 김통계입니다. 기억해 주세요.")
print("첫 번째 답:", 답1[:100])
print()

답2 = ask("제 이름이 뭐라고 했죠?")     # 새 요청 = 백지 상태 / a new request starts blank
print("두 번째 답:", 답2[:150])

In [ ]:
# (2) 이전 대화를 함께 보내면 -- 기억하는 것처럼 보입니다
# EN — (2) Send the history along, and it looks as though it remembers
messages = [
    {"role": "user",      "content": "제 이름은 김통계입니다. 기억해 주세요."},
    {"role": "assistant", "content": 답1},          # 모델이 앞서 한 말 / what the model said earlier
    {"role": "user",      "content": "제 이름이 뭐라고 했죠?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages)
print(response.choices[0].message.content)

> **핵심**
> 기억은 모델이 아니라 **우리 코드**가 만드는 것입니다.
> 대화가 길어지면 매번 보내는 양이 늘어나고, 결국 한계에 부딪힙니다.
> 이 문제를 **4주차(메모리)** 에서 본격적으로 다룹니다.

> **EN — the point**
> Memory is created by **our code**, not by the model. As the conversation
> grows, so does what we resend each time, and eventually it hits the limit.
> **Week 4 (memory)** takes this problem on properly.

---
## 5. 토큰 — 얼마나 썼는지 확인하기 · Tokens — checking what you spent

모델은 글자가 아니라 **토큰** 단위로 텍스트를 처리합니다.
응답 객체의 `usage` 를 보면 이번 호출에 몇 개를 썼는지 알 수 있습니다.

> **EN** — The model processes text in **tokens**, not characters. The `usage`
> field of the response object tells you how many this call consumed.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "중심극한정리를 세 문장으로 설명해 주세요."}],
)

u = response.usage
print("입력 토큰 :", u.prompt_tokens)
print("출력 토큰 :", u.completion_tokens)
print("합계      :", u.total_tokens)
print()
print(response.choices[0].message.content)

In [ ]:
# 한국어와 영어는 토큰 소비량이 다릅니다
# EN — Korean and English consume tokens at very different rates
문장들 = {
    "한국어": "표본의 크기가 커지면 표본평균의 분포는 정규분포에 가까워집니다.",
    "영어":   "As the sample size increases, the distribution of the sample mean approaches a normal distribution.",
}

for 언어, 문장 in 문장들.items():
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": 문장}],
        max_tokens=1,          # 답은 필요 없고 입력 토큰만 세고 싶으므로 / we only want the input count
    )
    print(f"{언어}: {r.usage.prompt_tokens:3d} 토큰   ({len(문장)} 글자)")

---
## 연습문제 · Exercises

제출하지 않습니다. 직접 해 보고 아래 해설과 비교하세요.

**문제 1.** `system` 메시지를 세 가지로 바꾸어 같은 질문을 던지고,
답의 **어조**와 **길이(글자 수)** 가 어떻게 달라지는지 표로 정리하시오.

**문제 2.** 위 토큰 비교에서 한국어가 영어보다 토큰을 더 많이 쓰는 이유는 무엇일까?
(힌트: 모델이 학습한 데이터의 언어 구성)

**문제 3.** `ask()` 함수에 `max_tokens=30` 을 넣어 호출해 보시오.
응답이 어떻게 되는가? 이런 일이 에이전트에서 일어나면 어떤 문제가 생기겠는가?

> **EN** — Not submitted. Try them yourself, then compare with the solutions.
>
> **1.** Ask the same question under three different `system` messages and
> tabulate how the **tone** and the **length (character count)** of the answer
> change.
>
> **2.** In the token comparison above, why does Korean use more tokens than
> English? (Hint: the language mix of the model's training data.)
>
> **3.** Call `ask()` with `max_tokens=30`. What happens to the response? What
> would go wrong if this happened inside an agent?

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 1.**
`system` 은 대화 전체에 적용되는 지시입니다. 어조뿐 아니라 답의 길이, 용어 수준,
형식(목록/문장)까지 크게 좌우합니다. 에이전트를 만들 때 `system` 은
"이 부품이 무슨 일을 하는가"를 정의하는 자리가 됩니다. 3주차에서 본격적으로 설계합니다.

**문제 2.**
토크나이저는 학습 데이터에서 자주 나온 문자열을 하나의 토큰으로 묶습니다.
대부분의 모델은 영어 텍스트를 훨씬 많이 학습했기 때문에, 영어 단어는 통째로 1토큰이
되는 경우가 많은 반면 한국어는 글자 단위로 잘게 쪼개집니다.
→ **같은 내용이라도 한국어 프롬프트가 컨텍스트를 더 많이 잡아먹습니다.**

**문제 3.**
문장이 중간에서 잘립니다. 에이전트에서는 이것이 치명적입니다.
모델의 출력을 코드로 파싱해야 하는데(3주차), JSON이 중간에 끊기면 파싱이 실패합니다.
`max_tokens` 는 "안전장치"이지 "길이 조절 손잡이"가 아닙니다.
길이를 줄이고 싶으면 프롬프트로 지시하십시오.

---

**EN — 1.** `system` is an instruction that applies to the entire conversation.
It governs not just tone but answer length, vocabulary level and format (list
versus prose). When you build an agent, `system` is where you define what a
component does. Week 3 designs this properly.

**EN — 2.** A tokenizer merges frequently seen strings into single tokens. Most
models have seen far more English text, so an English word is often one whole
token while Korean is split character by character. → **For the same content, a
Korean prompt eats considerably more context.**

**EN — 3.** The sentence is cut off mid-way. Inside an agent this is fatal: we
have to parse the model's output in code (week 3), and a JSON object that stops
half-way fails to parse. `max_tokens` is a SAFETY STOP, not a length dial. To
shorten an answer, instruct it in the prompt.

</details>

---
## 정리 · Summary

- 새 API는 **가장 단순한 요청부터** 보내서 진단한다
- `messages` 는 목록이고, 역할은 `system` / `user` / `assistant`
- **모델은 기억하지 못한다.** 기억은 우리 코드가 만든다
- 토큰을 항상 의식한다. 한국어는 더 비싸다

> **EN**
> - Diagnose a new API by sending **the simplest request first**
> - `messages` is a list; the roles are `system` / `user` / `assistant`
> - **The model does not remember.** Our code creates the memory
> - Always keep tokens in mind — Korean costs more

**다음 노트북** → [`02_chat_params.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week01/02_chat_params.ipynb)
temperature가 응답을 얼마나 흔드는지 직접 측정합니다.

> **Next notebook** — measure for yourself how much `temperature` shakes the output.